<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 2.2 - RAG Foundations: Embeddings + Retrieval + LLM

In this notebook, you will build a simple Retrieval-Augmented Generation (RAG) workflow over de-identified clinical notes prepared in Lab 1.

### What you will do
1) Load cleaned notes (Lab 1 output)
2) Embed notes using MiniLM (local embeddings)
3) Use FAISS to retrieve the most relevant notes for a query
4) Provide retrieved notes to a local LLM (Ollama) to extract a **dementia phenotype (Yes/No)**

### Why this matters
Dementia is often under-coded in structured diagnosis tables. RAG lets us search narrative notes for phenotyping signals that structured data may miss.

## 1. Prepare Notes Data for RAG

We will load the cleaned notes dataset from Lab 6 (deduplicated and restricted to the cohort window).

**Important:**
We will NOT use physician gold labels inside the retrieval or prompt context (to avoid leakage).
Gold labels are only used later for evaluation.

In [1]:
# -----------------------------------------------------------
# 1.1. Load Cleaned Notes + Evaluation Labels (Lab 6 Outputs)
# -----------------------------------------------------------
# Inputs:
# - lab6_clean_notes_baseline.parquet  (note-level dataset for RAG)
# - lab6_structured_dementia_eval.csv  (patient-level labels for evaluation only)
#
# Goal:
# - notes_df: used for embeddings + retrieval (NO gold labels attached)
# - eval_df: used later to evaluate RAG outputs
# -----------------------------------------------------------

from IPython.display import display, Markdown
from pathlib import Path
import pandas as pd

filepath = Path("data_files")

notes_df = pd.read_parquet(filepath / "lab1_clean_notes_baseline.parquet")
eval_df  = pd.read_csv(filepath / "lab1_structured_dementia_eval.csv",  dtype=str)

print("Notes shape:", notes_df.shape)
print("Eval shape :", eval_df.shape)

display(notes_df.head(10))

Notes shape: (2507, 7)
Eval shape : (90, 6)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,Physical therapy progress note for visit #2. T...
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,The patient was seen for follow-up regarding s...
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,This is a progress note for a 69-year-old woma...
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,Assessment & Plan:\n\nThe patient has a histor...
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This is an update on an 85-year-old man with a...
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,A message was left for the patient requesting ...
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient was seen for evaluation and manage...
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\nThe patient returned for fo...
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presented for a follow-up evaluati...


## 2. Embed and Store Clinical Notes in a FAISS Vector Store (In-Memory)

In this step, we embed full clinical notes and store them in a **FAISS** (Facebook AI Similarity Search
) vector store, which enables efficient similarity search. We use a lightweight transformer model (`MiniLM`) to convert each note into a semantic vector.

### Steps:
- **2.1**: Embed clinical notes using the `all-MiniLM-L6-v2` model from Hugging Face.
- **2.2**: View an embedded document along with its metadata and vector representation.

### Why Use This Approach?

Storing entire notes is useful when:
- You want to preserve the full clinical context for each patient.
- Your downstream use case (e.g., summarization or structured extraction) requires complete narrative input.
- The notes are concise enough to fit within the input limits of an LLM.

This method simplifies retrieval workflows by allowing you to work with whole documents rather than fragmented chunks.

<img src="./images/rag_full.png" alt="RAG Full" width="900">




In [10]:
# -----------------------------------------------------------
# 2.1. Embed Clinical Notes Using Local MiniLM Embeddings
# -----------------------------------------------------------
# This cell encodes each clinical note into a vector using a local
# transformer model and stores those embeddings in a FAISS index for
# fast similarity search.

# Model: `sentence-transformers/all-MiniLM-L6-v2`
# - Optimized for semantic similarity tasks
# - Lightweight and fast (384-dimensional vectors)
# - Runs fully offline
# -----------------------------------------------------------
import os, logging
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Prepare inputs for embedding: note text and relevant metadata
documents = notes_df["note_txt_deid"].tolist()
metadata = notes_df[["patient_num", "visit_id", "create_dts_shifted", "note_csn_id" ,"inpatient_note_type_dsc"]].to_dict(orient="records")

# Create a FAISS vector store from the documents
vectorstore = FAISS.from_texts(
    documents,
    embedding_model,
    metadatas=metadata
)

print(f"✅ Successfully embedded {len(documents)} clinical notes using MiniLM.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Successfully embedded 2507 clinical notes using MiniLM.


In [3]:
# -----------------------------------------------------------
# 2.2. View a Specific Embedded Document, Metadata, and Vector
# -----------------------------------------------------------
# Select an index (e.g., id = 5) to inspect the stored document.
# This cell shows the document text, associated metadata, and
# the corresponding FAISS embedding vector.
# -----------------------------------------------------------

id = 1  # You can change this index to view a different record

# Get (doc_id, Document) tuple from LangChain's docstore
doc_id, doc_example = list(vectorstore.docstore._dict.items())[id]

# Retrieve corresponding vector from FAISS
vector_example = vectorstore.index.reconstruct(id)

display(Markdown(f"### 🧾 Document ID: `{doc_id}`"))
display(Markdown(f"**Metadata:** `{doc_example.metadata}`"))

display(Markdown("**Document Text (First 500 characters):**"))
display(Markdown(f"```text\n{doc_example.page_content}...\n```"))

display(Markdown("**Embedded Vector (First 100):**"))
display(Markdown(f"```text\n{vector_example[:100]}\n```"))

### 🧾 Document ID: `3ffbd280-ae68-45c3-a28b-d6a875a75809`

**Metadata:** `{'patient_num': 'H111336931', 'visit_id': 6332144391, 'create_dts_shifted': Timestamp('2017-01-01 17:11:00'), 'note_csn_id': 2712892487, 'inpatient_note_type_dsc': 'Progress Notes'}`

**Document Text (First 500 characters):**

```text
The patient was seen for follow-up regarding systemic lupus erythematosus (SLE), approximately 18 months since her last visit. She has recently developed new skin lesions on her hands, which are asymptomatic but have progressed. A prior dermatology consultation resulted in treatment with topical steroids that provided minimal benefit. The patient reports that the color changes in her hands worsen with cold exposure, though she does not experience symptoms consistent with Raynaud's phenomenon.

Her review of systems is notable for chronic kidney disease, unchanged, and a history of SLE with intermittent mild thrombocytopenia. There has been no other organ involvement, and she remains physically active. Her medications include hydroxychloroquine 200 mg daily and a ProAir inhaler.

On examination, blood pressure was 147/67, pulse 63, and temperature 97. Head and neck exam was normal, with a clear oral cavity. A mild malar flush was present. Joint examination showed no active synovitis throughout the upper and lower extremities. Closer inspection of the hands revealed macular and papular eruptions, some blanching, also in the periungual (nailbed) area, resembling chilblain lesions.

The clinical impression is SLE with primarily cutaneous disease and new onset skin lesions suspicious for chilblain. These are typically asymptomatic and may occur during winter, resolving with warmer temperatures. Periungual erythema is observed, which can be a feature of cutaneous lupus, but currently, no changes to her hydroxychloroquine regimen are recommended. More potent immunosuppressants such as CellCept or azathioprine are not indicated given the mild symptoms.

The plan is for continued monitoring through the winter, with reassessment in the spring if lesions persist. A routine follow-up is scheduled in one year....
```

**Embedded Vector (First 100):**

```text
[-0.0768174  -0.02949139  0.01869638  0.0144869  -0.05379644 -0.05157546
  0.07288428  0.15090346 -0.03451969 -0.01810681  0.03019881  0.06657054
  0.04571561  0.03816699 -0.03774419  0.0266423   0.02520393 -0.06747247
 -0.00613535  0.06814315 -0.02541487 -0.04857438 -0.09446896 -0.03128262
 -0.00275252  0.083686   -0.02664507 -0.03719879 -0.01968771  0.07322513
 -0.02161509  0.04569168 -0.13132493  0.03562557  0.01340992  0.00439424
  0.00901844  0.03807427 -0.05234205  0.08022856  0.01317388 -0.09246907
 -0.06302503 -0.02808986 -0.01637342 -0.00092734 -0.0323302   0.09265915
  0.01971488 -0.01983921  0.02743189 -0.06529503  0.05908427 -0.03385709
 -0.02908773  0.00537916 -0.00723249 -0.10336778  0.02731097  0.00318129
 -0.0897335   0.02818906  0.02204301 -0.0387242   0.0217741  -0.02405902
  0.05450751 -0.1049254   0.08609073  0.04994913  0.08782806 -0.00617897
 -0.0055373   0.02657118  0.00188724  0.01481984 -0.01058711 -0.05633616
  0.00692143 -0.02460867  0.04485586  0.07760336  0.04053608  0.05829377
  0.07536598  0.00029566 -0.00888911 -0.02602716 -0.06382035  0.03034804
  0.00544917  0.01489581 -0.027769   -0.05994448  0.05429768  0.02710543
 -0.01360718 -0.02063933 -0.05628522  0.00648601]
```

## 3. Retrieving Clinical Notes with Similarity Score (RAG Retrieval)

In this section, we perform semantic search over embedded clinical notes using a FAISS vector store and a locally generated query vector. We use similarity scores to evaluate the relevance of each match to the query.

### Key Retrieval Steps:

1. **Embed a Query (Step 3.1)**
   - Converts a natural language question into a numerical vector using the same MiniLM model used to embed the notes.

2. **Similarity Search with Scores (Step 3.2)**
   - Retrieves the top-k clinical notes ranked by cosine similarity to the query.
   - Includes similarity scores for transparency and ranking.

3. **Score Threshold Filtering (Step 3.3)**
   - Filters out matches with low similarity scores.
   - Helps improve the precision and clinical relevance of the results.

### Why Use These Techniques?

Similarity search helps identify notes most relevant to a user-defined question or condition. Threshold filtering ensures:
- Only strong matches are considered for downstream tasks like summarization
- Noisy or unrelated content is excluded
- Each result can be justified based on a similarity score

<img src="./images/rag_retrieval.png" alt="RAG Retrieval" width="900">


In [4]:
# -----------------------------------------------------------
# 3.1. Embed a Query and Inspect Its Vector Representation
# -----------------------------------------------------------
# This step encodes a natural language query into a numerical vector
# using the same MiniLM model used for the clinical notes.
# This vector will be used to search for semantically similar notes.
# -----------------------------------------------------------

# Define a sample clinical query
query = "Which patients have documented dementia or Alzheimer disease?"

# Generate the embedding for the query
query_vector = embedding_model.embed_query(query)

# Display the vector and its shape
display(Markdown("### Vectorized Query"))
display(Markdown(f"`Query:` *{query}*"))
display(Markdown("**Embedding Vector (truncated):**"))
display(Markdown(f"```text\n{query_vector[:100]} ... [{len(query_vector)} dimensions]\n```"))


### Vectorized Query

`Query:` *Which patients have documented dementia or Alzheimer disease?*

**Embedding Vector (truncated):**

```text
[0.011906935833394527, 0.03944014757871628, -0.09752410650253296, 0.06022509187459946, -0.04667927324771881, 0.04867030680179596, -0.05022354796528816, 0.029064081609249115, -0.028790153563022614, -0.023942913860082626, 0.03695804253220558, 0.05865553021430969, -0.02685238979756832, 0.049717485904693604, -0.051887545734643936, 0.05872119963169098, -0.08355791121721268, 0.04715631902217865, 0.06059790030121803, 0.02870223857462406, -0.020331967622041702, 0.023083791136741638, 0.06421563029289246, -0.0038007257971912622, 0.02896592766046524, -0.021871929988265038, -0.05860956758260727, -0.086844801902771, 0.012490646913647652, 0.0435485802590847, 0.005426767282187939, 0.04009679704904556, -0.049031179398298264, 0.040420711040496826, -0.0029337992891669273, -0.042801544070243835, -0.037239041179418564, 0.07821102440357208, -0.045647356659173965, -0.03393026441335678, -0.012661042623221874, 0.016520781442523003, 0.07573212683200836, -0.0023874626494944096, -0.01474627386778593, -0.04000601917505264, -0.01247819047421217, -0.010556485503911972, -0.01922827586531639, -0.004239893052726984, -0.0685100182890892, -0.03398379310965538, 0.005000611301511526, 0.04402044042944908, 0.033297717571258545, 0.019468585029244423, 0.03023684397339821, -0.03426550328731537, -0.08062276989221573, 0.03611023724079132, -0.06497181951999664, -0.04285372421145439, -0.03309416025876999, 0.007066899444907904, -0.038591545075178146, 0.0902220755815506, -0.028005165979266167, -0.05437342822551727, -0.0321253165602684, -0.07484523206949234, -0.02471080794930458, -0.0031285732984542847, 0.029322747141122818, 0.06368803977966309, 0.013924732804298401, -0.01169080100953579, -0.035682786256074905, 0.0597379207611084, -0.01746251806616783, -0.13445661962032318, 0.0181600209325552, 0.03753931447863579, 0.04464970901608467, -0.004935245029628277, 0.11035894602537155, 4.173434717813507e-05, 0.0027889148332178593, 0.03659074753522873, -0.07208330184221268, -0.013018954545259476, 0.04452178627252579, -0.00509884487837553, -0.06256204098463058, -0.06569273769855499, -0.05408468097448349, -0.011294152587652206, 0.04169526323676109, 0.03355439007282257, -0.02642577514052391, 0.002572137862443924] ... [384 dimensions]
```

In [5]:
# -----------------------------------------------------------
# 3.2. Similarity Search (Top-K Results, No Filtering)
# -----------------------------------------------------------
# This cell performs a semantic similarity search using the embedded query,
# returning the top-k most similar clinical notes along with similarity scores.

# Score Interpretation (Euclidean Distance):
# - 0.00 – 0.30: Highly relevant
# - 0.30 – 0.50: Strong match
# - 0.50 – 0.70: Moderate match
# - 0.70 – 0.90: Low match
# - 0.90+: Minimal or irrelevant

# similarity_score = 1 / (1 + distance)  # Rescales Euclidean Distance into similarity-like score
# -----------------------------------------------------------

# Define number of top results
top_k = 10

# Run similarity search
results = vectorstore.similarity_search_with_score(query, k=top_k)

# Display header
display(Markdown(f"### 🔍 Top {top_k} Most Similar Clinical Notes"))

# Iterate and display each match
for i, (doc, score) in enumerate(results):
    display(Markdown(f"---\n**Result {i+1}**  \n- **Distance Score:** `{score:.4f}`  \n- **Patient Num:** `{doc.metadata.get('patient_num', 'N/A')}`  \n- **Visit ID:** `{doc.metadata.get('visit_id', 'N/A')}`\n\n**Note Preview:**\n```text\n{doc.page_content[:1200]}\n```"))


### 🔍 Top 10 Most Similar Clinical Notes

---
**Result 1**  
- **Distance Score:** `0.8074`  
- **Patient Num:** `H116088887`  
- **Visit ID:** `6387926043`

**Note Preview:**
```text
A recent outreach call was made to the patient to check in, and a voice mail message was left. Coordination was discussed with the social worker. Continued monitoring is recommended, given the established diagnosis of dementia.
```

---
**Result 2**  
- **Distance Score:** `0.8234`  
- **Patient Num:** `H112686715`  
- **Visit ID:** `6447865553`

**Note Preview:**
```text
A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia.
```

---
**Result 3**  
- **Distance Score:** `0.8853`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `6353841785`

**Note Preview:**
```text
Patient was evaluated during this admission. Plans previously outlined were reviewed and found appropriate. Examination was performed, and the care provided remains in alignment with prior assessments. The patient's history is notable for a prior diagnosis of dementia.
```

---
**Result 4**  
- **Distance Score:** `0.8872`  
- **Patient Num:** `H117326759`  
- **Visit ID:** `6432799125`

**Note Preview:**
```text
Medications were reviewed and reconciled on 9/23/17. Services were provided by an agency from 8/26/17 to 10/24/17. The patient's care is being managed with attention to medication safety, taking into account a prior diagnosis of dementia.
```

...

---
**Result 6**  
- **Distance Score:** `0.8945`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `6368409967`

**Note Preview:**
```text
The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided.
```

---
**Result 7**  
- **Distance Score:** `0.9007`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `6346678323`

**Note Preview:**
```text
The patient was seen and examined. On repeat examination, the lungs were clear without wheezes. The care plan established by the NP was reviewed and agreed upon. There is a history of a prior diagnosis of dementia as documented in previous notes.
```

---
**Result 8**  
- **Distance Score:** `0.9257`  
- **Patient Num:** `H117604331`  
- **Visit ID:** `6403897423`

**Note Preview:**
```text
Today, the patient was seen in person for a follow-up visit and to receive a flu vaccination. Reports from her daughter indicate that there have been no new concerns and her condition remains stable. The patient continues to receive assistance at home with her activities of daily living. The care team will remain available for ongoing support and plans to check in by phone in approximately 3 months. There is a history of cognitive impairment consistent with a highly confident diagnosis of dementia.
```

---
**Result 9**  
- **Distance Score:** `0.9263`  
- **Patient Num:** `H113694308`  
- **Visit ID:** `6469116677`

**Note Preview:**
```text
The patient is currently admitted for oncology care. Plans should be coordinated with the primary oncology team, who provides ongoing management. The patient has previously been under oncology follow-up. The team is available for consultation as needed. Given the patient's prior diagnosis of dementia, care plans should incorporate appropriate support and monitoring. Communication with the oncology consult service is advised if further guidance is required.
```

---
**Result 10**  
- **Distance Score:** `0.9282`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `6373843211`

**Note Preview:**
```text
The patient remains stable on the present medication regimen. Previous diagnosis of dementia continues to be managed effectively. No changes in treatment are necessary at this time.
```

In [6]:
# -----------------------------------------------------------
# 3.3. Filter Search Results by Similarity Score Threshold
# -----------------------------------------------------------
# This cell filters the top-K search results to keep only those
# with high similarity scores above a defined threshold.

# Similarity Score Threshold:
# - Only notes with scores ≥ threshold will be retained.
# - Higher scores = greater semantic similarity.
# -----------------------------------------------------------


from IPython.display import Markdown, display

threshold = 0.9  # Keep notes with score ≥ x

# Initialize an empty list to hold the filtered results
filtered_results = []

# Loop through each result (a tuple of Document and score)

for doc, distance in results:
    # Check if the similarity score meets the threshold
    if distance <= threshold:
        # If so, add it to the filtered list
        filtered_results.append((doc, distance))


# Summary
display(Markdown(f"### ✅ {len(filtered_results)} of {top_k} notes passed the distance threshold (<= {threshold})"))

# Show filtered results
for i, (doc, similarity) in enumerate(filtered_results):
    display(Markdown(
        f"---\n**Filtered Match {i+1}**  \n"
        f"- **Distance Score:** `{similarity:.4f}`  \n"
        f"- **Patient Num:** `{doc.metadata.get('patient_num', 'N/A')}`  \n"
        f"**Note Preview:**\n```text\n{doc.page_content[:1200]}\n```"
    ))


### ✅ 6 of 10 notes passed the distance threshold (<= 0.9)

---
**Filtered Match 1**  
- **Distance Score:** `0.8074`  
- **Patient Num:** `H116088887`  
**Note Preview:**
```text
A recent outreach call was made to the patient to check in, and a voice mail message was left. Coordination was discussed with the social worker. Continued monitoring is recommended, given the established diagnosis of dementia.
```

---
**Filtered Match 2**  
- **Distance Score:** `0.8234`  
- **Patient Num:** `H112686715`  
**Note Preview:**
```text
A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia.
```

---
**Filtered Match 3**  
- **Distance Score:** `0.8853`  
- **Patient Num:** `H116882795`  
**Note Preview:**
```text
Patient was evaluated during this admission. Plans previously outlined were reviewed and found appropriate. Examination was performed, and the care provided remains in alignment with prior assessments. The patient's history is notable for a prior diagnosis of dementia.
```

---
**Filtered Match 4**  
- **Distance Score:** `0.8872`  
- **Patient Num:** `H117326759`  
**Note Preview:**
```text
Medications were reviewed and reconciled on 9/23/17. Services were provided by an agency from 8/26/17 to 10/24/17. The patient's care is being managed with attention to medication safety, taking into account a prior diagnosis of dementia.
```

---
**Filtered Match 5**  
- **Distance Score:** `0.8916`  
- **Patient Num:** `H115124574`  
**Note Preview:**
```text
On 10/19/2017, the patient was admitted as an inpatient. Contact was successfully made with the patient and an email was sent to the inpatient case manager regarding the admission. The primary medical concern noted at the time of communication was congestive heart failure. Of note, there is a prior working diagnosis of dementia that may inform ongoing management considerations.
```

---
**Filtered Match 6**  
- **Distance Score:** `0.8945`  
- **Patient Num:** `H115124574`  
**Note Preview:**
```text
The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided.
```

## 4. Generating Structured Responses with an LLM (RAG Generation)

In this section, we take the clinical notes retrieved via semantic search and pass them into a Large Language Model (LLM) to generate structured, clinically meaningful responses. This is the final step in the **Retrieval-Augmented Generation (RAG)** pipeline.

### Key Steps:

1. **Creating a Prompt Template for LLM Querying (Step 4.1)**
   - Defines a reusable prompt structure for analyzing and summarizing clinical notes.
   - Ensures each response includes patient metadata and clear, structured outputs.

2. **Invoking LLM with Retrieved Context (Step 4.2)**
   - Inserts the top-retrieved clinical notes into the prompt.
   - Sends the prompt to a local LLM (e.g., Qwen2 via Ollama) for structured generation.
   - Returns a summary that directly answers the user’s medical query.

### Why This Matters

This step demonstrates how LLMs can synthesize information from real patient notes to produce:
- Patient-specific summaries
- Answered clinical questions
- Traceable outputs with structured identifiers

This capability is essential for use cases like clinical decision support, patient-facing summaries, or intelligent search interfaces.

<img src="./images/rag_generation.png" alt="RAG Generation" width="1250">


In [7]:
# -----------------------------------------------------------
# 4.1. Prompt Template: Dementia Phenotype Extraction (Yes/No)
# -----------------------------------------------------------
# We will ask the LLM to phenotype dementia using ONLY retrieved note text.
# No gold labels are included in the prompt.
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", f"""
You are a clinical phenotyping assistant.
Use ONLY the provided notes. Do NOT infer or invent.
Do NOT mention privacy/redaction/date shifting/removed identifiers.
"""),

    ("human", f"""
Retrieved notes:

{{retrieved_docs}}

Clinical question: {{query}}

For each relevant patient, provide:

Return a structured answer using bullet points:
1) Dementia Phenotype (Yes/No)
2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')
3) Rationale: 1–2 sentences grounded in the evidence
4) Confidence: low/medium/high

Rules for Dementia Phenotype:
- Answer 'Yes' ONLY if dementia is explicitly documented (e.g., 'dementia', 'Alzheimer',
  'vascular dementia', 'Lewy body dementia') or clearly stated as a prior diagnosis.
- If dementia is not explicitly mentioned, answer 'No' (do not infer from memory complaints alone).
""")
])

print(">> Prompt created and ready to use.")

>> Prompt created and ready to use.


In [8]:
# -----------------------------------------------------------
# 4.2. Run the Phenotype Prompt on ALL Filtered Retrieved Notes
# -----------------------------------------------------------
# We include identifiers (patient_num, visit_id, note_csn_id) for traceability.
# We pass all filtered results (after similarity threshold) into the prompt.
# -----------------------------------------------------------

from IPython.display import display, Markdown
from langchain_ollama import ChatOllama

# Safety: limit how much text per note we pass to the model
MAX_CHARS_PER_NOTE = 2500

if len(filtered_results) == 0:
    display(Markdown("### No notes passed the similarity threshold."))
else:
    # Build retrieved docs text with identifiers
    retrieved_blocks = []
    for rank, (doc, score) in enumerate(filtered_results, start=1):
        meta = doc.metadata or {}

        patient_num = meta.get("patient_num", "N/A")
        visit_id = meta.get("visit_id", "N/A")
        note_csn_id = meta.get("note_csn_id", "N/A")
        note_type = meta.get("inpatient_note_type_dsc", "N/A")
        create_dts = meta.get("create_dts_shifted", "N/A")

        note_text = (doc.page_content or "").strip()
        note_text = note_text[:MAX_CHARS_PER_NOTE]

        retrieved_blocks.append(
            f"---\n"
            f"[Retrieved Note #{rank} | distance={score:.4f}]\n"
            f"patient_num={patient_num} | visit_id={visit_id} | note_csn_id={note_csn_id}\n"
            f"note_type={note_type} | create_dts_shifted={create_dts}\n\n"
            f"{note_text}\n"
        )

    retrieved_context = "\n".join(retrieved_blocks)

    # Show what will be sent (optional but useful in workshop)
    display(Markdown(f"### Retrieved notes sent to the LLM: {len(filtered_results)}"))
    display(Markdown(retrieved_context[:3000] + ("\n\n*(preview truncated)*" if len(retrieved_context) > 3000 else "")))

    # Fill prompt
    final_prompt = prompt_template.format(
        retrieved_docs=retrieved_context,
        query=query
    )

    # Invoke local model (deterministic)
    model = ChatOllama(model="gemma3:12b", temperature=0.0)
    response = model.invoke(final_prompt)

    display(Markdown("### 📋 LLM Output: Dementia Phenotype from Retrieved Notes"))
    display(Markdown(response.content))

### Retrieved notes sent to the LLM: 6

---
[Retrieved Note #1 | distance=0.8074]
patient_num=H116088887 | visit_id=6387926043 | note_csn_id=3286513763
note_type=Progress Notes | create_dts_shifted=2017-11-03 15:39:00

A recent outreach call was made to the patient to check in, and a voice mail message was left. Coordination was discussed with the social worker. Continued monitoring is recommended, given the established diagnosis of dementia.

---
[Retrieved Note #2 | distance=0.8234]
patient_num=H112686715 | visit_id=6447865553 | note_csn_id=3925256583
note_type=Progress Notes | create_dts_shifted=2017-09-11 14:36:00

A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia.

---
[Retrieved Note #3 | distance=0.8853]
patient_num=H116882795 | visit_id=6353841785 | note_csn_id=2927596265
note_type=Progress Notes | create_dts_shifted=2017-06-11 11:52:00

Patient was evaluated during this admission. Plans previously outlined were reviewed and found appropriate. Examination was performed, and the care provided remains in alignment with prior assessments. The patient's history is notable for a prior diagnosis of dementia.

---
[Retrieved Note #4 | distance=0.8872]
patient_num=H117326759 | visit_id=6432799125 | note_csn_id=3768256165
note_type=Progress Notes | create_dts_shifted=2017-09-23 14:37:00

Medications were reviewed and reconciled on 9/23/17. Services were provided by an agency from 8/26/17 to 10/24/17. The patient's care is being managed with attention to medication safety, taking into account a prior diagnosis of dementia.

---
[Retrieved Note #5 | distance=0.8916]
patient_num=H115124574 | visit_id=6382479701 | note_csn_id=3225668549
note_type=Progress Notes | create_dts_shifted=2017-10-19 11:39:00

On 10/19/2017, the patient was admitted as an inpatient. Contact was successfully made with the patient and an email was sent to the inpatient case manager regarding the admission. The primary medical concern noted at the time of communication was congestive heart failure. Of note, there is a prior working diagnosis of dementia that may inform ongoing management considerations.

---
[Retrieved Note #6 | distance=0.8945]
patient_num=H115124574 | visit_id=6368409967 | note_csn_id=3075125799
note_type=Progress Notes | create_dts_shifted=2017-08-18 14:02:00

The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided.


### 📋 LLM Output: Dementia Phenotype from Retrieved Notes

Here's a structured answer identifying patients with documented dementia, based solely on the provided notes:

*   **Patient ID:** H116088887
    *   **Dementia Phenotype:** Yes
    *   **Evidence:** "established diagnosis of dementia", "Continued monitoring is recommended, given the established diagnosis of dementia."
    *   **Rationale:** The note explicitly states an "established diagnosis of dementia," indicating a confirmed diagnosis.
    *   **Confidence:** High

*   **Patient ID:** H112686715
    *   **Dementia Phenotype:** Yes
    *   **Evidence:** "consistent with a working diagnosis of dementia"
    *   **Rationale:** The note indicates a "working diagnosis of dementia," suggesting a diagnosis has been made.
    *   **Confidence:** Medium

*   **Patient ID:** H116882795
    *   **Dementia Phenotype:** Yes
    *   **Evidence:** "prior diagnosis of dementia"
    *   **Rationale:** The note mentions a "prior diagnosis of dementia," confirming a previous diagnosis.
    *   **Confidence:** High

*   **Patient ID:** H117326759
    *   **Dementia Phenotype:** Yes
    *   **Evidence:** "prior diagnosis of dementia"
    *   **Rationale:** The note explicitly mentions a "prior diagnosis of dementia."
    *   **Confidence:** High

*   **Patient ID:** H115124574
    *   **Dementia Phenotype:** Yes
    *   **Evidence:** "prior working diagnosis of dementia", "history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia"
    *   **Rationale:** The note mentions a "prior working diagnosis of dementia" and a "history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia."
    *   **Confidence:** Medium